# ToSDR Experiment
Given a site name or URL, fetch its ToSDR grade + all human-annotated clauses via the public API.
No auth required. We also cross-check by scraping the live ToS page with Scrapling.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../backend"))

import httpx
import json
from pprint import pprint

## Step 1 — Search ToSDR for a service

`/search/v4/?query=NAME` returns a list of matching services with their IDs and grades.
We take the top hit.

In [12]:
TOSDR_BASE = "https://api.tosdr.org"

def search_service(query: str) -> list[dict]:
    """Search ToSDR for a service by name. Returns a list of matches."""
    url = f"{TOSDR_BASE}/search/v4/"
    resp = httpx.get(url, params={"query": query}, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    # Response shape: {"parameters": {"services": [...]}}
    services = data.get("parameters", {}).get("services", [])
    return services

# Try it
results = search_service("discord")
print(f"Found {len(results)} result(s)\n")
for svc in results[:10]:
    grade = svc.get("rating", {})
    if isinstance(grade, dict):
        grade = grade.get("letter", "?")
    print(f"  id={svc['id']:>6}  grade={grade}  name={svc['name']}")

Found 10 result(s)

  id=   536  grade=D  name=Discord
  id=  4965  grade=N/A  name=DiscordTop
  id=  8181  grade=N/A  name=DiscordHub
  id= 11117  grade=N/A  name=Discord Me
  id=  2513  grade=N/A  name=DiscordList
  id=  7932  grade=D  name=DiscordRep 
  id=  5614  grade=C  name=Discord.Club
  id=  4575  grade=N/A  name=onDiscord.xyz
  id= 10012  grade=N/A  name=Discord Bot List 
  id=   585  grade=N/A  name=Reswitched discord


## Step 2 — Fetch full service details + all annotated points

`/service/v3/?id=ID` returns the complete service object including every individual clause
with its classification (`good`, `bad`, `alert`, `neutral`, `blocker`) and human summary.

In [14]:
def get_service_details(service_id: int) -> dict:
    """Fetch full service object. /service/v3/ returns data at root, not under 'parameters'."""
    resp = httpx.get(f"{TOSDR_BASE}/service/v3/", params={"id": service_id}, timeout=15)
    resp.raise_for_status()
    return resp.json()   # root IS the service object

def get_tosdr(service_name: str) -> dict | None:
    hits = search_service(service_name)
    if not hits:
        print(f"No ToSDR entry found for '{service_name}'")
        return None
    top = hits[0]
    print(f"Using top hit: id={top['id']}  name={top['name']}")
    return get_service_details(top["id"])

discord = get_tosdr("discord")
# print(f"\nKeys: {list(discord.keys())}")
# print(f"rating type: {type(discord['rating'])}  value: {discord['rating']}")
# print(f"urls sample: {discord['urls'][:3]}")
# print(f"documents[0]: {discord['documents'][0]}")
# print(f"\n--- First full point (raw) ---")
# print(json.dumps(discord["points"][0], indent=2) if discord.get("points") else "no points")
print(discord)

Using top hit: id=536  name=Discord
{'id': 536, 'is_comprehensively_reviewed': False, 'name': 'Discord', 'updated_at': '2026-03-06T10:55:44.301431', 'created_at': '2018-06-20T12:22:06.538763', 'slug': 'discord', 'rating': 'D', 'urls': ['discord.com', 'discord.gg', 'discord.co', 'discordapp.io', 'discord.dev', 'discord.new', 'discord.media', 'discord.design', 'discordapp.com', 'discordapp.net', 'dis.gd', 'discord.gift', 'discordstatus.com', 'discordcdn.com', 'airhorn.solutions', 'airhornbot.com', 'bigbeans.solutions', 'watchanimeattheoffice.com'], 'image': 'https://s3.tosdr.org/logos/536.png', 'documents': [{'id': 15037, 'name': 'Paid Services Terms', 'url': 'https://discord.com/terms/paid-services-terms', 'updated_at': '2026-03-06T10:28:16.818907', 'created_at': '2023-03-29T16:20:20.967511'}, {'id': 3005, 'name': 'Community Guidelines', 'url': 'https://discord.com/guidelines', 'updated_at': '2026-03-06T10:29:42.264990', 'created_at': '2020-11-02T09:53:53.634141'}, {'id': 985, 'name': '

## Step 3 — Parse and display the data cleanly

Extract: overall grade, ToS URLs, and each annotated point with its classification + summary.

In [15]:
CLASSIFICATION_EMOJI = {
    "good":    "✅",
    "neutral": "➖",
    "bad":     "⚠️",
    "alert":   "🚨",
    "blocker": "🛑",
}

def parse_tosdr(data: dict) -> dict:
    # rating is a plain string e.g. "D", not a dict
    grade = data.get("rating") or "?"

    # urls is a list of domain strings — actual document URLs are in documents[]
    doc_urls = [d["url"] for d in data.get("documents", []) if d.get("url")]

    # points: each has title, status (approval), analysis (classification), quoteDoc, etc.
    # We'll dump the raw first point to learn the exact field names after running
    points_raw = data.get("points", [])
    points = []
    for p in points_raw:
        # classification field — check analysis vs status after seeing raw output
        classification = (
            p.get("analysis")          # may be "good"/"bad"/"alert" etc
            or p.get("classification")
            or p.get("case", {}).get("classification") if isinstance(p.get("case"), dict) else None
            or "unknown"
        )
        points.append({
            "classification": classification,
            "title":          p.get("title", ""),
            "summary":        p.get("description", "") or p.get("tldr", "") or p.get("title", ""),
            "source_url":     p.get("source", ""),
            "quote":          p.get("quoteDoc", "") or p.get("quote", ""),
        })

    return {
        "name":      data.get("name", "?"),
        "grade":     grade,
        "doc_urls":  doc_urls,
        "points":    points,
    }


def display_tosdr(data: dict) -> None:
    parsed = parse_tosdr(data)
    print(f"{'='*60}")
    print(f"  Service  : {parsed['name']}")
    print(f"  Grade    : {parsed['grade']}")
    print(f"  Documents: {len(parsed['doc_urls'])}")
    for u in parsed["doc_urls"]:
        print(f"             {u}")
    print(f"  Points   : {len(parsed['points'])} annotated clauses")
    print(f"{'='*60}\n")

    from collections import defaultdict
    grouped = defaultdict(list)
    for p in parsed["points"]:
        grouped[p["classification"]].append(p)

    # Print known classifications first, then anything unexpected
    known = ["blocker", "alert", "bad", "neutral", "good"]
    all_keys = known + [k for k in grouped if k not in known]
    for cls in all_keys:
        items = grouped.get(cls, [])
        if not items:
            continue
        emoji = CLASSIFICATION_EMOJI.get(cls, "•")
        print(f"\n{emoji}  {cls.upper()} ({len(items)})")
        print("-" * 50)
        for p in items:
            print(f"  • {p['title']}")
            if p["summary"] and p["summary"] != p["title"]:
                summary = p["summary"][:200] + ("..." if len(p["summary"]) > 200 else "")
                print(f"    {summary}")

display_tosdr(discord)

  Service  : Discord
  Grade    : D
  Documents: 5
             https://discord.com/terms/paid-services-terms
             https://discord.com/guidelines
             https://discord.com/privacy
             https://discord.com/terms
             https://discord.com/terms/cookie-policy
  Points   : 15 annotated clauses


•  GENERATED THROUGH THE ANNOTATE VIEW (14)
--------------------------------------------------
  • Instead of asking directly, this Service will assume your consent merely from your usage.
  • Your account can be suspended for several reasons
  • You can delete your content from this service
  • Your account can be deleted without prior notice and without a reason
  • Information is gathered about you through third parties
  • Extra data may be collected about you through promotions
  • This service is only available to users over 13 years old
  • You authorise the service to charge a credit card supplied on re-occurring basis
  • Some personal data may be kept for bus

## Step 4 — Try all services from our eval corpus

Checks which of our 8 corpus platforms are on ToSDR and what grades they have.

In [16]:
import time

CORPUS_NAMES = ["discord", "spotify", "openai", "reddit", "netflix", "github", "notion", "substack"]

corpus_results = {}
for name in CORPUS_NAMES:
    try:
        hits = search_service(name)
    except httpx.HTTPStatusError as e:
        print(f"  {name:<12} — HTTP error: {e.response.status_code}, skipping")
        time.sleep(3)
        continue

    if not hits:
        print(f"  {name:<12} — NOT FOUND on ToSDR")
    else:
        top = hits[0]
        rating = top.get("rating", {})
        grade = rating.get("letter", "?") if isinstance(rating, dict) else str(rating or "?")
        print(f"  {name:<12} id={top['id']:<6} grade={grade}  matched_name='{top['name']}'")
        corpus_results[name] = top

    time.sleep(1.5)  # ToSDR rate-limits aggressively — stay polite

  discord      id=536    grade=D  matched_name='Discord'
  spotify      id=225    grade=D  matched_name='Spotify'
  openai       id=7108   grade=D  matched_name='OpenAI'
  reddit       id=194    grade=E  matched_name='Reddit'
  netflix      id=185    grade=D  matched_name='Netflix'
  github       id=297    grade=B  matched_name='GitHub'
  notion       id=1817   grade=C  matched_name='Notion'
  substack     id=6612   grade=C  matched_name='substack'


## Step 5 — Deep dive on one service: full clause list

Pick any slug from the corpus and see every annotated clause with its classification.

In [17]:
FOCUS = "spotify"   # ← change to any corpus slug

if FOCUS in corpus_results:
    service_id = corpus_results[FOCUS]["id"]
    details = get_service_details(service_id)
    display_tosdr(details)

    print("\n--- Raw JSON of first point (all available fields) ---")
    raw_points = details.get("points", [])
    if raw_points:
        print(json.dumps(raw_points[0], indent=2, default=str))
    else:
        print("no points returned")
else:
    print(f"{FOCUS} not in corpus_results — run Step 4 first")

  Service  : Spotify
  Grade    : C
  Documents: 7
             https://www.spotify.com/us/legal/end-user-agreement/
             https://www.spotify.com/us/legal/voice-controls/
             https://www.spotify.com/us/legal/end-user-agreement/
             https://www.spotify.com/us/legal/intellectual-property-policy/
             https://www.spotify.com/us/legal/user-guidelines/
             https://www.spotify.com/us/legal/paid-subscription-terms/
             https://www.spotify.com/us/legal/privacy-policy/
  Points   : 5 annotated clauses


•  GENERATED THROUGH THE ANNOTATE VIEW (1)
--------------------------------------------------
  • This service does not guarantee that it or the products obtained through it meet the users' expectations or requirements

•  CREATED BY DOCBOT VERSION V3 (4)
--------------------------------------------------
  • Your account can be suspended for several reasons
  • You maintain ownership of your content
  • You have the right to leave this service